# Palo Alto Networks — Career Intelligence & Retention Opportunity Platform
## Unsupervised Machine Learning, Feature Engineering, and Career Path Clustering

This notebook provides the complete exploratory data analysis (EDA), mathematical feature derivations, unsupervised clustering validation (K-Means vs. Hierarchical), PCA dimensionality reduction, and prescriptive career archetype interpretation.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'

### 1. Ingest Raw Workforce Dataset

In [2]:
data_path = '../data/raw/Palo Alto Networks(1).csv'
if not os.path.exists(data_path):
    data_path = 'Palo Alto Networks(1).csv'

df_raw = pd.read_csv(data_path)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

### 2. Feature Engineering: Career Velocity & Stagnation KPIs

We formulate mathematical derived indicators:
1. **Promotion Gap Ratio**: $\frac{YSLP}{YAC + 1}$
2. **Role Stagnation Index**: $\frac{YICR}{YAC + 1}$
3. **Training Intensity Score**: $\frac{\text{Training}}{YAC + 1}$
4. **Manager Stability Indicator**: $\frac{YWCM}{YICR + 1}$
5. **Promotion Gap Risk Score (0-100)**: Continuous composite stagnation score
6. **Retention Opportunity Index (ROI)**: Proactive retention prioritization index

In [3]:
df = df_raw.copy()
EPS = 1.0

# Derived KPIs
df['PromotionGapRatio'] = (df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + EPS)).round(3)
df['RoleStagnationIndex'] = (df['YearsInCurrentRole'] / (df['YearsAtCompany'] + EPS)).round(3)
df['TrainingIntensityScore'] = (df['TrainingTimesLastYear'] / (df['YearsAtCompany'] + EPS)).round(3)
df['ManagerStabilityIndicator'] = (df['YearsWithCurrManager'] / (df['YearsInCurrentRole'] + EPS)).round(3)
df['CareerVelocity'] = (df['JobLevel'] / (df['TotalWorkingYears'] + EPS)).round(3)

# Composite Promotion Gap Risk Score
raw_risk = (
    np.clip(df['YearsSinceLastPromotion'] / 10.0, 0, 1.0) * 35.0 +
    np.clip(df['RoleStagnationIndex'], 0, 1.0) * 25.0 +
    np.clip(df['YearsInCurrentRole'] / 8.0, 0, 1.0) * 25.0 +
    (1.0 - (df['JobSatisfaction'] / 4.0)) * 15.0
)
df['PromotionGapRiskScore'] = np.clip(raw_risk, 0, 100).round(1)

# Retention Opportunity Index
df['RetentionOpportunityIndex'] = np.clip(
    (1 - df['Attrition']) * 25.0 + 
    (df['PerformanceRating'] / 4.0) * 25.0 + 
    (df['JobInvolvement'] / 4.0) * 15.0 + 
    (df['PromotionGapRiskScore'] / 100.0) * 35.0,
    0, 100
).round(1)

print("Engineered Features Created Successfully.")
df[['PromotionGapRatio', 'RoleStagnationIndex', 'PromotionGapRiskScore', 'RetentionOpportunityIndex']].describe()

### 3. Unsupervised Clustering & Optimal K Evaluation
We test K-Means clustering across $K \in [3, 7]$ and measure Silhouette, Calinski-Harabasz, and Davies-Bouldin metrics.

In [4]:
clustering_features = [
    'TotalWorkingYears', 'YearsAtCompany', 'YearsInCurrentRole', 
    'YearsSinceLastPromotion', 'YearsWithCurrManager', 'JobLevel',
    'PromotionGapRatio', 'RoleStagnationIndex', 'TrainingIntensityScore', 
    'ManagerStabilityIndicator', 'CareerVelocity', 'PercentSalaryHike'
]

X = df[clustering_features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []
for k in range(3, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = km.fit_predict(X_scaled)
    results.append({
        'K': k,
        'Silhouette': silhouette_score(X_scaled, labels),
        'Calinski_Harabasz': calinski_harabasz_score(X_scaled, labels),
        'Davies_Bouldin': davies_bouldin_score(X_scaled, labels)
    })

eval_df = pd.DataFrame(results)
eval_df

### 4. Optimal Clustering (K=5) & Principal Component Analysis (PCA)

In [5]:
kmeans_optimal = KMeans(n_clusters=5, random_state=42, n_init=15)
df['ClusterID'] = kmeans_optimal.fit_predict(X_scaled)

pca = PCA(n_components=3, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['PCA1'] = pca_coords[:, 0]
df['PCA2'] = pca_coords[:, 1]
df['PCA3'] = pca_coords[:, 2]

print("PCA Explained Variance:", pca.explained_variance_ratio_)
print(f"Cumulative Variance Captured: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")

### 5. Career Archetype Interpretation & Centroids

In [6]:
centroid_summary = df.groupby('ClusterID')[clustering_features + ['PromotionGapRiskScore', 'RetentionOpportunityIndex', 'MonthlyIncome', 'Attrition']].mean().round(2)
centroid_summary.T

### 6. PCA 2D Cluster Visualization

In [7]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='ClusterID', palette='viridis', alpha=0.85, s=40)
plt.title("Workforce Career Space (PCA 2D Projection)", fontsize=14, fontweight='bold')
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster ID", loc="upper right")
plt.tight_layout()
plt.show()